<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">

  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">

  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">
      Trabajo de Fin de Máster
    </div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">
      Clasificador taxonómico de boletines oficiales españoles
    </div>
  </div>

  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>

  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** — ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** — Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** — Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** — Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

| § | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo |
| **1** | **Schema de output** | `ClassifierOutput`, enums y reglas de negocio |
| **2** | **Ground truth** | Construcción del dataset etiquetado manualmente |
| **3** | **Agente clasificador** | System prompt, agent con Pydantic AI, run single |
| **4** | **Baseline** | Experimento 0 — claude-sonnet-4-6 sobre ground truth |
| **5** | **Experimentos de ablación** | Few-shot, `act_type` y campo `technologies` |
| **6** | **Modelos locales** | Experimento 4 — modelos 7B via LM Studio |
| **7** | **Análisis y conclusiones** | Comparativa de métricas, falsos positivos, siguientes pasos |

---

## §0. Setup

Cargamos las variables de entorno (la API key de Gemini vive en `.env`, nunca en el código) e importamos las librerías del proyecto.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

load_dotenv(find_dotenv())

True

In [2]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# Modelo local via LM Studio
# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

print(f"Modelo: {LM_STUDIO_MODEL} via LM Studio (localhost:1234)")

Modelo: qwen/qwen3.5-9b via LM Studio (localhost:1234)


/tmp/ipykernel_22141/210525475.py:8: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [3]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    print("LM Studio operativo.")
    print(f"Modelos disponibles: {modelos}")
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"  {LM_STUDIO_MODEL} listo")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde — ¿está el servidor arrancado en localhost:1234?")

LM Studio operativo.
Modelos disponibles: ['qwen/qwen3.5-9b', 'google/gemma-3n-e4b', 'gemma-4-e2b-it', 'google/gemma-3-4b', 'gemma-4-e4b-it', 'text-embedding-nomic-embed-text-v1.5']
  qwen/qwen3.5-9b listo


---

## §1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) — resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) — multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) — multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [4]:
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput

In [5]:
# Ejemplo válido: resolución de DIA con tecnología fotovoltaica
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning=(
        "'se formula la declaración de impacto ambiental' → DIA en resolución. "
        "'Planta Solar Fotovoltaica' → fotovoltaica."
    ),
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

# Ejemplo que viola el invariante: is_relevant=False con procedures no vacío
print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False,
        act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP],
        technologies=[],
        confidence=0.5,
        reasoning="Prueba de invariante.",
    )
except Exception as e:
    print(f"  ValidationError capturado → {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "DIA"
  ],
  "technologies": [
    "fotovoltaica"
  ],
  "confidence": 0.98,
  "reasoning": "'se formula la declaración de impacto ambiental' → DIA en resolución. 'Planta Solar Fotovoltaica' → fotovoltaica."
}

Violación de invariante:
  ValidationError capturado → Value error, is_relevant=False con procedures != []


---

## §2. Ground truth

El ground truth tiene tres capas que se construyen en orden:

| Capa | Qué etiqueta | Cómo | Estado |
|------|-------------|------|--------|
| **1 — N1 determinista** | `act_type` | Reglas de primer token sobre `description` | Esta sección |
| **2 — N2 por LLM** | `procedures`, `technologies`, `is_relevant` | Agente Pydantic AI sobre muestra estratificada | §3–§4 |
| **3 — Revisión manual** | Correcciones y casos límite | Inspección humana de los desacuerdos | Post-baseline |

Esta sección implementa la **Capa 1**: inferencia determinista de `act_type` mediante reglas de primer token. El resultado se usará como señal de entrada al prompt del LLM en §3 y como columna de estratificación para el muestreo del ground truth en §4.

In [6]:
import html
import re

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"

df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")

Corpus: 65,201 registros · 19 boletines


In [7]:
def preprocess_description(desc: str, bulletin: str) -> str:
    desc = desc.strip()
    # BOCM: "Tema\n– Tipo de acto..."
    if "\n–" in desc:
        desc = desc.split("\n–", 1)[1].strip()
    elif "\n-" in desc:
        desc = desc.split("\n-", 1)[1].strip()
    # BOCA: "Organismo.- Tipo de acto..."
    if bulletin == "boca" and ".-" in desc:
        desc = desc.split(".-", 1)[1].strip()
    # BOE topónimos: descripción es solo ciudad en mayúsculas
    if re.match(r"^[A-ZÁÉÍÓÚÜÑ/\s]+$", desc) and len(desc.split()) <= 4:
        return "__TOPONIMO__"
    # BOE subastas AEAT
    if desc.upper().startswith(("U.R.", "E.R.", "SUMA GESTIÓN", "ORGANISMO AUTÓNOMO DE HACIENDA")):
        return "__SUBASTA_AEAT__"
    return desc

In [8]:
_N1_MAP = [
    (r"corrección de errat",             ActType.CORRECCION_ERRORES),
    (r"corrección de error",             ActType.CORRECCION_ERRORES),
    (r"rectificación",                   ActType.CORRECCION_ERRORES),
    (r"real decreto",                    ActType.REAL_DECRETO),
    (r"orden foral",                     ActType.ORDEN),
    (r"información pública",             ActType.INFORMACION_PUBLICA),
    (r"exposición pública",              ActType.INFORMACION_PUBLICA),
    (r"trámite de información",          ActType.INFORMACION_PUBLICA),
    (r"resolución",                      ActType.RESOLUCION),
    (r"anuncio",                         ActType.ANUNCIO),
    (r"orden",                           ActType.ORDEN),
    (r"decreto foral",                   ActType.DECRETO),
    (r"decreto",                         ActType.DECRETO),
    (r"acuerdo",                         ActType.ACUERDO),
    (r"aprobación",                      ActType.APROBACION),
    (r"extracto",                        ActType.EXTRACTO),
    (r"convenio",                        ActType.CONVENIO),
    (r"adenda",                          ActType.CONVENIO),
    (r"solicitud",                       ActType.SOLICITUD),
    (r"modificación",                    ActType.MODIFICACION),
    (r"edicto",                          ActType.EDICTO),
    (r"notificación",                    ActType.NOTIFICACION),
    (r"notificaciones",                  ActType.NOTIFICACION),
    (r"recaudación ejecutiva",           ActType.NOTIFICACION),
    (r"propuesta de resolución",         ActType.RESOLUCION),
    (r"bases",                           ActType.CONVOCATORIA),
    (r"convocatoria",                    ActType.CONVOCATORIA),
    (r"nombramiento",                    ActType.RESOLUCION),
    (r"delegación",                      ActType.RESOLUCION),
    (r"emplazamiento",                   ActType.NOTIFICACION),
    (r"citación",                        ActType.NOTIFICACION),
    (r"diligencia",                      ActType.NOTIFICACION),
    (r"cédula",                          ActType.NOTIFICACION),
    (r"requerimiento",                   ActType.NOTIFICACION),
    (r"sala primera",                    ActType.OTROS),
    (r"sala segunda",                    ActType.OTROS),
    (r"__toponimo__",                    ActType.OTROS),
    (r"__subasta_aeat__",                ActType.OTROS),
]


def inferir_act_type(description: str, bulletin: str) -> ActType:
    desc_clean = preprocess_description(description, bulletin)
    text = desc_clean.lower().strip()
    for pattern, act_type in _N1_MAP:
        if text.startswith(pattern):
            return act_type
    return ActType.OTROS

In [9]:
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

print("Distribución N1 inferida:\n")
dist = df["act_type_n1"].value_counts()
total = len(df)
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

otros = (df["act_type_n1"] == ActType.OTROS).sum()
cobertura = (1 - otros / total) * 100
print(f"\nCobertura N1: {cobertura:.1f}%  ({otros:,} registros en OTROS)")

Distribución N1 inferida:

  resolución                23,360  (35.8%)
  anuncio                   16,357  (25.1%)
  otros                      7,410  (11.4%)
  orden                      3,198  (4.9%)
  aprobación                 2,910  (4.5%)
  notificación               2,565  (3.9%)
  edicto                     1,486  (2.3%)
  información_pública        1,467  (2.2%)
  extracto                   1,385  (2.1%)
  acuerdo                    1,320  (2.0%)
  corrección_errores           954  (1.5%)
  decreto                      808  (1.2%)
  convocatoria                 761  (1.2%)
  convenio                     686  (1.1%)
  real_decreto                 251  (0.4%)
  solicitud                    173  (0.3%)
  modificación                 110  (0.2%)

Cobertura N1: 88.6%  (7,410 registros en OTROS)


### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre **88.6%** del corpus con reglas deterministas.
El 11.4% restante cae en `OTROS` por tres motivos distintos con soluciones conocidas:

| Grupo | Volumen | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.600 | Sin contenido textual real — irrecuperable sin PDF | Clasificar como clase propia `NO_INFERIBLE` en v2 |
| BOCM/BOCA residual | ~300 | Variantes de cabecera no contempladas | Ampliar reglas de pre-procesamiento |
| RRHH sin tipo explícito | ~2.000 | `RELACIÓN`, `LISTA`, `OFERTA`, `REGISTRO`, `ASPIRANTES`... — el tipo de acto no aparece en la descripción | LLM zero-shot puede inferirlo desde el contenido; viable en v2 |

El tercer grupo es el más interesante: los registros de RRHH con descripción como
*"Relación definitiva de aspirantes admitidos..."* o *"Lista provisional de admitidos..."*
tienen suficiente contenido semántico para que un LLM infiera `resolución` o `anuncio`.
Esto está documentado aquí como trabajo pendiente para la extensión v2 del clasificador.

### Muestreo estratificado del ground truth

Esta celda extrae los **500 registros** que se etiquetarán manualmente como ground truth. El muestreo es estratificado por keyword para garantizar representación de todos los procedimientos N2 — sin estratificación, los ~2.400 registros relevantes quedarían subrepresentados frente a los ~62.600 negativos.

Las etiquetas LLM (`is_relevant_pred`, `procedures_pred`, etc.) se añadirán en §4 cuando el agente esté operativo.

In [10]:
import random
random.seed(42)

# Estrategia: buscar keywords en description para cada procedimiento N2
keywords = {
    "DIA":     ["declaración de impacto ambiental"],
    "AAP":     ["autorización administrativa previa"],
    "AAC":     ["autorización de construcción", "autorización administrativa de construcción"],
    "AAP_AAC": ["previa y de construcción"],
    "AAU":     ["autorización ambiental unificada"],
    "IIA":     ["informe de impacto ambiental"],
    "AAI":     ["autorización ambiental integrada"],
    "IAE":     ["ambiental estratégic"],
    "DUP":     ["utilidad pública"],
}

cuotas = {
    "DIA": 60, "AAP": 60, "AAC": 50, "AAP_AAC": 40,
    "AAU": 40, "IIA": 40, "AAI": 30, "IAE": 30,
    "DUP": 30,
}
N_NEGATIVOS = 120

sampled_ids = set()
frames = []

for grupo, kws in keywords.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    # Excluir registros ya seleccionados
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas[grupo], len(pool))
    sample = pool.sample(n, random_state=42)
    sample = sample.copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<10} pool={len(pool):>5,}  sampled={n}")

# Negativos: registros sin ninguna keyword relevante
all_kws = [kw for kws in keywords.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=42).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_gt = pd.concat(frames, ignore_index=True)
print(f"\nTotal muestreado: {len(df_gt)} registros")

  DIA        pool=  286  sampled=60
  AAP        pool=1,016  sampled=60
  AAC        pool=  514  sampled=50
  AAP_AAC    pool=  243  sampled=40
  AAU        pool=  182  sampled=40
  IIA        pool=  521  sampled=40
  AAI        pool=  257  sampled=30
  IAE        pool=  319  sampled=30
  DUP        pool=  635  sampled=30

Total muestreado: 500 registros


In [11]:
import os

PATH_GT = "../data/ground_truth/ground_truth_v1.csv"
os.makedirs(os.path.dirname(PATH_GT), exist_ok=True)

# Guardar solo las columnas necesarias para la anotación
cols = ["bulletin", "description", "act_type_n1", "grupo_muestreo"]
df_gt[cols].to_csv(PATH_GT, index=False)

print(f"Ground truth guardado en {PATH_GT}")
print(f"Columnas: {cols}")
print(df_gt["grupo_muestreo"].value_counts().to_string())

Ground truth guardado en ../data/ground_truth/ground_truth_v1.csv
Columnas: ['bulletin', 'description', 'act_type_n1', 'grupo_muestreo']
grupo_muestreo
NEGATIVO    120
DIA          60
AAP          60
AAC          50
AAP_AAC      40
AAU          40
IIA          40
AAI          30
IAE          30
DUP          30


### Columnas pendientes — se rellenan en §4

El CSV actual contiene solo metadatos y el tipo de acto pre-computado.
Cuando el agente esté operativo (§4), se añadirán estas columnas:

| Columna | Quién la rellena |
|---|---|
| `is_relevant_pred` | Agente LLM |
| `procedures_pred` | Agente LLM |
| `technologies_pred` | Agente LLM |
| `act_type_pred` | Agente LLM (Config Baseline) |
| `confidence` | Agente LLM |
| `reasoning` | Agente LLM |
| `is_relevant_gt` | Revisión manual |
| `procedures_gt` | Revisión manual |
| `technologies_gt` | Revisión manual |
| `needs_review` | Flag automático: confidence < 0.8 |

---

## §3. Agente clasificador

El agente se compara en dos configuraciones que se evaluarán en §5:

| Configuración | Input al LLM | Qué infiere el LLM |
|---|---|---|
| **Baseline** | `description` + `bulletin` | `act_type`, `procedures`, `technologies`, `is_relevant` |
| **+N1** | `description` + `bulletin` + `act_type` pre-computado | `procedures`, `technologies`, `is_relevant` (con contexto extra) |

La hipótesis es que darle el `act_type` ya clasificado por reglas reduce los falsos positivos en `procedures` — el LLM no tiene que adivinar si es una resolución o un anuncio y puede centrarse en identificar el procedimiento.

In [12]:
SYSTEM_PROMPT = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles (BOE, BOJA, DOGC, etc.).
Tu tarea es identificar si una publicación pertenece al dominio de autorizaciones ambiental-energéticas,
que representa aproximadamente el 3,7% del corpus total.

## Procedimientos N2 (campo `procedures`)

| Código | Descripción |
|--------|-------------|
| DIA    | Declaración de Impacto Ambiental — acto formal por el que el organismo formula la DIA de un proyecto |
| AAP    | Autorización Administrativa Previa — primer paso de la autorización de instalaciones de generación |
| AAC    | Autorización Administrativa de Construcción — habilita el inicio de obras de la instalación |
| AAU    | Autorización Ambiental Unificada — autorización ambiental integrada usada en Andalucía, Extremadura y Navarra |
| IIA    | Informe de Impacto Ambiental — evaluación simplificada (no es una declaración formal) |
| AAI    | Autorización Ambiental Integrada — permiso IPPC/IED para instalaciones industriales de gran impacto |
| IAE    | Informe Ambiental Estratégico — evaluación ambiental de planes y programas, no de proyectos individuales |
| DUP    | Declaración de Utilidad Pública — puede acompañar a AAP/AAC pero no equivale a ninguna de ellas |

## Tecnologías N3 (campo `technologies`)

| Código           | Descripción |
|------------------|-------------|
| fotovoltaica     | Plantas solares fotovoltaicas, huertos solares, instalaciones FV |
| eólica           | Parques eólicos, aerogeneradores, instalaciones de energía eólica |
| almacenamiento   | Sistemas BESS, baterías de almacenamiento energético |
| hibridación      | Instalaciones híbridas que combinan dos o más tecnologías de generación |
| hidroeléctrica   | Centrales hidroeléctricas, minicentrales, aprovechamientos hidráulicos |
| biogás_biometano | Plantas de biogás, biometano, digestión anaerobia |
| biomasa          | Plantas de biomasa, cogeneración con biomasa, bioenergía sólida |
| hidrógeno        | Instalaciones de hidrógeno verde, electrolizadores |
| línea_eléctrica  | Líneas de alta y media tensión, subestaciones, infraestructura de evacuación |
| gas_natural      | Gasoductos, instalaciones de gas natural, regasificadoras |
| petróleo         | Refinerías, oleoductos, instalaciones petrolíferas |

## Reglas de negocio críticas

1. `is_relevant=True` **solo** si identificas al menos un procedimiento N2 en la lista anterior.
2. **AAU ≠ DIA** — no colapsar. AAU aparece principalmente en BOJA (Andalucía), DOE (Extremadura) y BON (Navarra).
3. **AAP + AAC juntas** — "autorización administrativa previa y de construcción" en el mismo acto → `[AAP, AAC]`.
4. **IIA ≠ DIA** — "informe de impacto ambiental" es una evaluación simplificada, no una declaración formal.
5. **AAI ≠ DIA** — "autorización ambiental integrada" es permiso IPPC/IED industrial; solo es DIA si el texto menciona explícitamente "declaración de impacto ambiental".
6. **IAE** aplica exclusivamente a planes y programas, no a proyectos individuales.
7. **DUP** puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola.
8. El campo `reasoning` debe citar el **fragmento exacto** del texto que dispara cada etiqueta asignada.
""".strip()

In [13]:
agent = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT,
)

In [14]:
_GAZETTE_TO_AMBITO = {
    "boe": "estatal",
    "madridambiental": "local",
    # autonómico es el valor por defecto para el resto
}


async def clasificar(description: str, bulletin: str, use_n1_context: bool = False) -> ClassifierOutput:
    """
    Clasifica una publicación de boletín oficial.

    Args:
        description: Texto de la publicación (ya con html.unescape aplicado)
        bulletin: Código del boletín en minúsculas (ej. 'boja', 'boe')
        use_n1_context: Si True, añade el act_type pre-computado como contexto (Config +N1)
    """
    n0 = _GAZETTE_TO_AMBITO.get(bulletin, "autonómico")
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"

    if use_n1_context:
        act_type_precomputed = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_precomputed.value}"

    result = await agent.run(user_msg)
    return result.output

In [15]:
casos = [
    # Caso 1 — DIA clara
    ("Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluación "
     "Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "
     "Planta Solar Fotovoltaica Los Llanos, en la provincia de Cáceres.", "doe"),

    # Caso 2 — AAP+AAC conjunta
    ("Resolución de 5 de febrero de 2025, de la Dirección General de Política Energética, "
     "por la que se otorga autorización administrativa previa y de construcción para el "
     "Parque Eólico Sierra Norte, de 48 MW, en Salamanca.", "boe"),

    # Caso 3 — AAU (no DIA)
    ("Resolución de 18 de enero de 2025, de la Delegación Territorial de Medio Ambiente, "
     "por la que se otorga autorización ambiental unificada para la planta de biogás "
     "Valdecorneja, en Ávila.", "boja"),

    # Caso 4 — No relevante
    ("Resolución de 3 de marzo de 2025, de la Universidad de Salamanca, por la que se "
     "convoca concurso-oposición para cubrir plazas de profesor ayudante doctor.", "bocyl"),

    # Caso 5 — IIA (no DIA)
    ("Resolución de 21 de febrero de 2025, de la Dirección General de Medio Natural, "
     "por la que se formula el informe de impacto ambiental del proyecto de línea "
     "eléctrica subterránea de 132 kV en Zaragoza.", "boa"),
]

for i, (desc, bulletin) in enumerate(casos, 1):
    print(f"{'='*60}")
    print(f"CASO {i} — {bulletin.upper()}")
    print(f"Descripción: {desc[:80]}...")
    print()
    resultado = await clasificar(desc, bulletin)
    print(resultado.model_dump_json(indent=2))
    print()

CASO 1 — DOE
Descripción: Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluaci...

{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "DIA"
  ],
  "technologies": [
    "fotovoltaica"
  ],
  "confidence": 0.98,
  "reasoning": "El texto menciona explícitamente \"se formula la declaración de impacto ambiental\" del proyecto \"Planta Solar Fotovoltaica Los Llanos\", lo que identifica el procedimiento DIA y la tecnología fotovoltaica con alta certeza."
}

CASO 2 — BOE
Descripción: Resolución de 5 de febrero de 2025, de la Dirección General de Política Energéti...

{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "AAP",
    "AAC"
  ],
  "technologies": [
    "eólica"
  ],
  "confidence": 0.98,
  "reasoning": "El texto menciona explícitamente la \"autorización administrativa previa y de construcción\" (AAP y AAC) para un \"Parque Eólico Sierra Norte\", lo que confirma la presencia del procedimiento N2 de generación

---

## §4. Baseline

Esta sección ejecuta el **Experimento 0 — Baseline** (sin contexto N1) sobre los 500 registros del ground truth. El agente clasifica cada publicación solo con `description` y `bulletin`, sin saber el `act_type` pre-computado.

El resultado es `../results/baseline_results.csv` — un CSV con las predicciones del agente listo para calcular métricas en §7. La sección está preparada pero comentada: se ejecuta cuando el modelo esté operativo (mañana con LM Studio).

In [15]:
import asyncio
import json
from pathlib import Path

In [16]:
async def clasificar_async(
    description: str,
    bulletin: str,
    use_n1_context: bool = False,
) -> dict:
    """Versión async de clasificar(). Devuelve dict listo para CSV."""
    n0 = _GAZETTE_TO_AMBITO.get(bulletin, "autonómico")
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"

    if use_n1_context:
        act_type_precomputed = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_precomputed.value}"

    result = await agent.run(user_msg)
    output = result.output

    return {
        "is_relevant_pred": output.is_relevant,
        "act_type_pred": output.act_type.value,
        "procedures_pred": json.dumps([p.value for p in output.procedures], ensure_ascii=False),
        "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
        "confidence": output.confidence,
        "reasoning": output.reasoning,
    }

In [17]:
from tqdm.asyncio import tqdm_asyncio


async def run_experiment(
    df_input: pd.DataFrame,
    use_n1_context: bool = False,
    concurrency: int = 5,
    output_path: str = "../results/baseline_results.csv",
) -> pd.DataFrame:
    """
    Ejecuta el agente sobre todos los registros del DataFrame.
    Guarda progreso incremental cada 50 registros por si se interrumpe.
    """
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)
    results = []

    async def process_row(row):
        async with semaphore:
            try:
                pred = await clasificar_async(
                    row["description"], row["bulletin"], use_n1_context
                )
            except Exception as e:
                pred = {
                    "is_relevant_pred": None,
                    "act_type_pred": None,
                    "procedures_pred": "[]",
                    "technologies_pred": "[]",
                    "confidence": None,
                    "reasoning": f"ERROR: {str(e)[:100]}",
                }
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando")

    df_results = pd.DataFrame(results)

    # Guardar CSV final
    df_results.to_csv(output_path, index=False)
    print(f"\nResultados guardados en {output_path}")
    print(f"Errores: {df_results['reasoning'].str.startswith('ERROR').sum()}")

    return df_results

In [18]:
df_gt_full = pd.read_csv("../data/ground_truth/ground_truth_v1.csv")

# Muestra proporcional de ~100 registros manteniendo la distribución por grupo
total = len(df_gt_full)
df_gt_100 = pd.concat([
    g.sample(min(len(g), max(1, round(len(g) * 100 / total))), random_state=42)
    for _, g in df_gt_full.groupby("grupo_muestreo")
]).reset_index(drop=True)

print(f"Muestra de {len(df_gt_100)} registros:")
print(df_gt_100["grupo_muestreo"].value_counts().to_string())

df_baseline_100 = await run_experiment(
    df_gt_100,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/baseline_100_qwen9b.csv",
)

print(df_baseline_100[["grupo_muestreo", "procedures_pred", "confidence"]].head(10))


Muestra de 100 registros:
grupo_muestreo
NEGATIVO    24
AAP         12
DIA         12
AAC         10
AAP_AAC      8
AAU          8
IIA          8
AAI          6
DUP          6
IAE          6


Clasificando: 100%|██████████| 100/100 [2:03:52<00:00, 74.32s/it] 


Resultados guardados en ../results/baseline_100_qwen9b.csv
Errores: 1
  grupo_muestreo        procedures_pred  confidence
0            AAC                     []        0.95
1            AAC  ["AAP", "AAC", "DUP"]        0.98
2            AAC  ["AAP", "AAC", "DIA"]        0.98
3            AAC         ["AAP", "AAC"]        0.95
4            AAC         ["AAP", "AAC"]        1.00
5            AAC         ["AAP", "AAC"]        0.98
6            AAC                     []         NaN
7            AAC         ["AAP", "AAC"]        0.95
8            AAC                     []        0.95
9            AAC         ["AAP", "AAC"]        0.98


### Nota sobre concurrencia y modelos locales

El parámetro `concurrency` controla cuántas llamadas al agente se hacen en paralelo.
- **Modelos locales (LM Studio)**: usar `concurrency=1` o `concurrency=2` — los modelos locales no ganan nada con más paralelismo y pueden saturarse
- **Modelos cloud (Gemini, Groq)**: usar `concurrency=5` o superior según los rate limits del tier

Si el experimento se interrumpe, el CSV parcial se pierde. Para experimentos largos,
descomentar el guardado incremental cada 50 registros en `run_experiment`.

---

## §7. Análisis y resultados

Esta sección evalúa el **Experimento Baseline**: Qwen 3.5 9B en modo zero-shot, sin contexto N1, sobre los 100 registros anotados manualmente.

El ground truth (`ground_truth_100_anotado.csv`) contiene las etiquetas humanas `is_relevant_gt`, `procedures_gt` y `technologies_gt`. Las predicciones (`baseline_100_qwen9b.csv`) fueron generadas por el agente en §4.

Se calculan tres niveles de métricas:

| Nivel | Campo | Métrica |
|-------|-------|---------|
| **N0** | `is_relevant` | Precisión, Recall, F1 binario |
| **N2** | `procedures` | F1 por etiqueta + macro/weighted + exact match sobre relevantes |
| **N3** | `technologies` | F1 por etiqueta (solo etiquetas con support > 0) |

In [ ]:
import json
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

df_gt = pd.read_csv("../data/ground_truth/ground_truth_100_anotado.csv")
df_pred = pd.read_csv("../results/baseline_100_qwen9b.csv")

# Los CSVs tienen los mismos registros en el mismo orden (misma pipeline de muestreo)
# Se combinan por posición; se usan description+bulletin como comprobación de alineación
assert (df_gt["description"].values == df_pred["description"].values).all(), \
    "Los registros no están alineados — revisar orden de los CSVs"

df = df_gt[["bulletin", "description", "grupo_muestreo",
            "is_relevant_gt", "procedures_gt", "technologies_gt"]].copy()
df["is_relevant_pred"]  = df_pred["is_relevant_pred"].values
df["procedures_pred"]   = df_pred["procedures_pred"].values
df["technologies_pred"] = df_pred["technologies_pred"].values
df["confidence"]        = df_pred["confidence"].values
df["reasoning"]         = df_pred["reasoning"].values

print(f"Registros para evaluar: {len(df)}")
print(f"Errores de predicción (None): {df['is_relevant_pred'].isna().sum()}")

In [ ]:
def parse_labels(value) -> set:
    """Convierte 'AAP,AAC' o '["AAP","AAC"]' a set."""
    if pd.isna(value) or str(value).strip() == '':
        return set()
    v = str(value).strip()
    if v.startswith('['):
        try:
            return set(json.loads(v))
        except Exception:
            pass
    return set(x.strip() for x in v.split(',') if x.strip())

In [ ]:
y_true = df["is_relevant_gt"].astype(bool)
y_pred = df["is_relevant_pred"].fillna(False).astype(bool)

print("── is_relevant ──────────────────────────")
print(f"  Precision : {precision_score(y_true, y_pred):.3f}")
print(f"  Recall    : {recall_score(y_true, y_pred):.3f}")
print(f"  F1        : {f1_score(y_true, y_pred):.3f}")
print()

tp = ( y_true  &  y_pred).sum()
fp = (~y_true  &  y_pred).sum()
fn = ( y_true  & ~y_pred).sum()
tn = (~y_true  & ~y_pred).sum()
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")

In [ ]:
N2_LABELS = ["DIA", "AAP", "AAC", "AAU", "IIA", "AAI", "IAE", "DUP"]

results_n2 = {}
for label in N2_LABELS:
    y_true_l = df.apply(lambda r: label in parse_labels(r["procedures_gt"]),   axis=1)
    y_pred_l = df.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)

    tp = ( y_true_l &  y_pred_l).sum()
    fp = (~y_true_l &  y_pred_l).sum()
    fn = ( y_true_l & ~y_pred_l).sum()

    p   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1  = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    results_n2[label] = {"precision": p, "recall": r, "f1": f1,
                         "support": int(y_true_l.sum()), "tp": tp, "fp": fp, "fn": fn}

print("── N2 · Procedimientos ──────────────────────────────────────────")
print(f"{'Label':<8} {'P':>6} {'R':>6} {'F1':>6} {'Support':>8} {'TP':>4} {'FP':>4} {'FN':>4}")
print("─" * 60)
for label, m in results_n2.items():
    print(f"{label:<8} {m['precision']:>6.3f} {m['recall']:>6.3f} {m['f1']:>6.3f} "
          f"{m['support']:>8} {m['tp']:>4} {m['fp']:>4} {m['fn']:>4}")

total_support = sum(m["support"] for m in results_n2.values())
macro_f1    = sum(m["f1"] for m in results_n2.values()) / len(N2_LABELS)
weighted_f1 = (sum(m["f1"] * m["support"] for m in results_n2.values()) / total_support
               if total_support > 0 else 0.0)
print("─" * 60)
print(f"{'Macro-F1':<8} {macro_f1:>6.3f}")
print(f"{'Weighted':<8} {weighted_f1:>6.3f}")

In [ ]:
exact = df.apply(
    lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]),
    axis=1,
)
relevantes_mask = df["is_relevant_gt"].astype(bool)

print(f"Exact match N2 (solo relevantes): {exact[relevantes_mask].mean():.3f}"
      f"  ({exact[relevantes_mask].sum()}/{relevantes_mask.sum()})")
print(f"Exact match N2 (todos):           {exact.mean():.3f}"
      f"  ({exact.sum()}/{len(df)})")

In [ ]:
N3_LABELS = ["fotovoltaica", "eólica", "almacenamiento", "hibridación",
             "hidroeléctrica", "biogás_biometano", "biomasa", "hidrógeno",
             "línea_eléctrica", "gas_natural", "petróleo"]

results_n3 = {}
for label in N3_LABELS:
    y_true_l = df.apply(lambda r: label in parse_labels(r["technologies_gt"]),   axis=1)
    y_pred_l = df.apply(lambda r: label in parse_labels(r["technologies_pred"]), axis=1)
    support = int(y_true_l.sum())
    if support == 0:
        continue
    tp = ( y_true_l &  y_pred_l).sum()
    fp = (~y_true_l &  y_pred_l).sum()
    fn = ( y_true_l & ~y_pred_l).sum()
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    results_n3[label] = {"precision": p, "recall": r, "f1": f1, "support": support}

print("── N3 · Tecnologías ─────────────────────────────────────────────")
print(f"{'Label':<20} {'P':>6} {'R':>6} {'F1':>6} {'Support':>8}")
print("─" * 50)
for label, m in results_n3.items():
    print(f"{label:<20} {m['precision']:>6.3f} {m['recall']:>6.3f} {m['f1']:>6.3f} {m['support']:>8}")

if results_n3:
    macro_f1_n3 = sum(m["f1"] for m in results_n3.values()) / len(results_n3)
    print("─" * 50)
    print(f"{'Macro-F1':<20} {macro_f1_n3:>6.3f}")

In [ ]:
print("── Errores N2 ───────────────────────────────────────────────────")
errores = df[~exact & relevantes_mask].reset_index(drop=True)
print(f"Total errores en relevantes: {len(errores)}")
print()
for i, row in errores.iterrows():
    print(f"#{i} | {row['bulletin'].upper()} | grupo={row['grupo_muestreo']}")
    print(f"  GT  : {row['procedures_gt']}")
    print(f"  PRED: {row['procedures_pred']}")
    print(f"  CONF: {row['confidence']}")
    print(f"  DESC: {str(row['description'])[:100]}...")
    print(f"  REASON: {str(row['reasoning'])[:120]}")
    print()

### Resumen de métricas — Experimento Baseline

| Métrica | Valor |
|---------|-------|
| **N0 is_relevant — Precision** | — |
| **N0 is_relevant — Recall** | — |
| **N0 is_relevant — F1** | — |
| **N2 Macro-F1** | — |
| **N2 Weighted-F1** | — |
| **N2 Exact match (relevantes)** | — |
| **N3 Macro-F1** | — |

> Modelo: Qwen 3.5 9B · Configuración: zero-shot, sin contexto N1 · n=100 registros anotados manualmente